# 🎬 ReAgent-V: Composed Video Retrieval (CoVR) Benchmark

**Bài toán**: Truy vấn video từ kho 4.886 video bằng Ảnh + Prompt text theo kiến trúc Two-Stage Agentic Retrieval.

**Pipeline**:
1. **Coarse Search** (CLIP): Lọc nhanh Top-N ứng viên
2. **Agentic Reranker** (LLaVA): Phân tích và xếp hạng tinh
3. **Adaptive Loop** (Memory Bank + Critic): Tự sửa chiến lược nếu điểm thấp

**Dataset**: WebVid-CoVR (`/kaggle/input/datasets/nta212/webvid-covr/`)

## ⚙️ Bước 0: Cài đặt môi trường

> ⚠️ **QUAN TRỌNG**: Không cài `flash-attn` qua `pip install` thông thường — nó sẽ biên dịch C++ tốn hàng GB RAM và gây crash kernel Kaggle. Notebook này dùng **pre-built wheel** thay thế.

In [ ]:
import subprocess, sys, os

# --- Buoc 0a: Cai dat transformers va accelerate tuong thich chuan voi LLaVA ---
print('Installing compatible packages...')
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==4.40.0',
    'accelerate==0.29.3',
    'bitsandbytes',
    'decord',
    'opencv-python-headless',
    'einops==0.6.1',
    'einops-exts==0.0.4',
    'timm==0.9.16',
    'ffmpeg-python',
    'networkx',
    'easydict',
    'soundfile',
    'librosa',
    'faiss-gpu',
], check=False)
print('Packages installed successfully.')

# --- Buoc 0b: Cai flash-attn tu pre-built wheel (KHONG compile tu source) ---
print('Checking flash-attn pre-built wheel...')
FLASH_WHEEL = (
    'https://github.com/Dao-AILab/flash-attention/releases/download/'
    'v2.5.7/flash_attn-2.5.7+cu122torch2.1cxx11abiFALSE-cp310-cp310-linux_x86_64.whl'
)
r = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', FLASH_WHEEL],
    capture_output=True, text=True
)
if r.returncode == 0:
    print('flash-attn: OK (pre-built wheel)')
else:
    print('flash-attn wheel incompatible -> using SDPA/eager attention fallback (OK for inference).')

print('Environment setup complete!')

## 📂 Bước 1: Clone repo ReAgent-V và cài LLaVA

In [ ]:
import os, sys, glob

REPO_ROOT = '/kaggle/working/EvoAgent'
REPO_DIR  = '/kaggle/working/EvoAgent/ReAgent-V'

# 1. Clone repo
if not os.path.exists(REPO_ROOT):
    os.system(f'git clone --depth=1 https://github.com/nta2112/EvoAgent.git {REPO_ROOT}')
    print('Repo cloned successfully.')
else:
    print('Repo already exists.')

# 2. Cai dat LLaVA-NeXT package truc tiep tu GitHub
print('Installing LLaVA-NeXT...')
os.system('pip install -q --no-deps git+https://github.com/LLaVA-VL/LLaVA-NeXT.git')

# 3. Tat breakpoint pdb va sua loi import neu co trong LLaVA
for p in glob.glob('/usr/local/lib/python*/dist-packages/llava/model/builder.py'):
    with open(p, 'r', encoding='utf-8') as f:
        c = f.read()
    c = c.replace('import pdb;pdb.set_trace()', 'pass').replace('import pdb; pdb.set_trace()', 'pass')
    with open(p, 'w', encoding='utf-8') as f:
        f.write(c)

# 4. Them duong dan module vao sys.path va chuyen working dir
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print(f'Working dir: {os.getcwd()}')
print('Setup completed!')

## 🔧 Bước 2: Khai báo tất cả đường dẫn

In [ ]:
import os

# =====================================================================
# PATHS — Dataset tu Kaggle (da upload san tai nta212/webvid-covr)
# =====================================================================
CSV_PATH  = '/kaggle/input/datasets/nta212/webvid-covr/WebVid_COVR_dataset/webvid8m-covr_test.csv'
VIDEO_DIR = '/kaggle/input/datasets/nta212/webvid-covr/WebVid_COVR_dataset/train'

# =====================================================================
# PATHS — Model weights (tai tu dong tu HuggingFace)
# =====================================================================
CLIP_MODEL   = 'openai/clip-vit-large-patch14-336'
WHISPER      = 'openai/whisper-base'
LLAVA        = 'lmms-lab/LLaVA-Video-7B-Qwen2'
MODELS_CACHE = '/kaggle/working/models'

# =====================================================================
# PATHS — Output files
# =====================================================================
INDEX_PATH   = '/kaggle/input/datasets/nta212/webvid-covr/covr_corpus_index.pt'
RESULTS_PATH = '/kaggle/working/eval_results.json'

# =====================================================================
# HYPERPARAMETERS (TOI UU TOC DO & DO CHINH XAC CAO)
# =====================================================================
NUM_SAMPLES      = 21     # 20 mau de danh gia chuan xac
TOP_K            = 10     # Xep hang Top-10
TOP_N_COARSE     = 20     # Quet 20 ung vien tu CLIP de bat duoc Ground Truth
MAX_ITERATIONS   = 3      # 1 vong rerank de chay nhanh nhat (~10-15s / query)
REWARD_THRESHOLD = 0.85   # Nguong diem Critic de dung som
ALPHA            = 0.50   # 35% Image + 65% Text

# Verify
print('CSV exists      :', os.path.exists(CSV_PATH))
print('Video dir exists:', os.path.exists(VIDEO_DIR))
print(f'Config: NUM_SAMPLES={NUM_SAMPLES}, TOP_K={TOP_K}, TOP_N_COARSE={TOP_N_COARSE}, MAX_ITERATIONS={MAX_ITERATIONS}, ALPHA={ALPHA}')




## 🚀 Bước 3: Load ReAgent-V và kiểm tra Dataset

In [ ]:
from ReAgentV import ReAgentV
from ReAgentV_utils.video_processor.covr_loader import load_covr_annotations, get_covr_query

# Kiểm tra annotation CSV
df_all = load_covr_annotations(CSV_PATH, VIDEO_DIR)
print(f'Total valid query triplets: {len(df_all)}')
print(df_all[['edit', 'pth1', 'pth2']].head(3))

In [ ]:
# Khoi tao he thong ReAgent-V (load CLIP + Whisper + LLaVA)
import torch
path_dict = {
    'clip_model_path'   : CLIP_MODEL,
    'clip_cache_dir'    : MODELS_CACHE,
    'whisper_model_path': WHISPER,
    'whisper_cache_dir' : MODELS_CACHE,
    'llava_model_path'  : LLAVA,
    'llava_cache_dir'   : MODELS_CACHE,
    'max_memory'        : {0: '11GiB', 1: '11GiB', 'cpu': '24GiB'},
    'attn_implementation': 'sdpa',
    'torch_dtype'       : torch.float16,
}
qa_system = ReAgentV.load_default(path_dict)
print('ReAgent-V system ready.')

## 📦 Bước 4: Tạo Video Index (chạy 1 lần, ~15-20 phút)

> **Bỏ qua cell này nếu file `covr_corpus_index.pt` đã tồn tại.**

In [ ]:
import os

if os.path.exists(INDEX_PATH):
    print(f'Index already exists: {INDEX_PATH} — skipping.')
else:
    print('Building corpus index...')
    os.system(
        f'python covr_indexer.py'
        f' --video_dir       {VIDEO_DIR}'
        f' --output_path     {INDEX_PATH}'
        f' --clip_model_path {CLIP_MODEL}'
        f' --clip_cache_dir  {MODELS_CACHE}'
        f' --batch_size      64'
        f' --device          cuda'
    )
    print('Done.')

In [ ]:
corpus_embeddings, corpus_paths = qa_system.load_corpus_index(INDEX_PATH, video_base_dir=VIDEO_DIR)
print(f"Corpus: {corpus_embeddings.shape[0]} videos | Dim: {corpus_embeddings.shape[1]}")
if len(corpus_paths) > 0:
    print(f"Sample video path: {corpus_paths[0]} (Exists: {os.path.exists(corpus_paths[0])})")


## 🔍 Bước 5: Demo truy vấn 1 mẫu

In [ ]:
from IPython.display import display
import os

SAMPLE_IDX = 0  # Thay số này để thử các mẫu khác nhau

df_sample = load_covr_annotations(CSV_PATH, VIDEO_DIR)
row = df_sample.iloc[SAMPLE_IDX]
query_image, query_text, gt_video_path = get_covr_query(row)

print('=' * 60)
print(f'Sample Index     : {SAMPLE_IDX}')
print(f'Edit Instruction : {query_text}')
print(f'Query video      : {row["pth1"]}')
print(f'Ground Truth     : {row["pth2"]}')
print('=' * 60)

if query_image is not None:
    display(query_image.resize((320, 180)))
    print('Query Image (middle frame of reference video)')

In [ ]:
top_k_results = qa_system.adaptive_covr_retrieval(
    query_image       = query_image,
    query_text        = query_text,
    corpus_embeddings = corpus_embeddings,
    corpus_paths      = corpus_paths,
    top_k             = TOP_K,
    top_n_coarse      = TOP_N_COARSE,
    max_iterations    = MAX_ITERATIONS,
    reward_threshold  = REWARD_THRESHOLD,
)

print('\n' + '=' * 60)
print(f'TOP-{TOP_K} RETRIEVAL RESULTS')
print('=' * 60)
gt_norm = os.path.normpath(gt_video_path)
for rank, (vpath, score, verdict) in enumerate(top_k_results, 1):
    hit = '  ✓ GROUND TRUTH HIT' if os.path.normpath(vpath) == gt_norm else ''
    print(f'  Rank {rank}: [{verdict:12s}] score={score:.4f} | {os.path.basename(vpath)}{hit}')
print('=' * 60)

## 📊 Bước 6: Chạy Benchmark (Recall@K và MRR)

In [ ]:
import json, time
from tqdm.notebook import tqdm

df_eval = load_covr_annotations(CSV_PATH, VIDEO_DIR, num_samples=NUM_SAMPLES)
total   = len(df_eval)

def recall_at_k(ranks, k, n):
    return sum(1 for r in ranks if r is not None and r <= k) / n

def mrr(ranks, n):
    return sum(1.0 / r for r in ranks if r is not None) / n

clip_ranks, agent_ranks, per_query = [], [], []
t0 = time.time()

for i, row in tqdm(df_eval.iterrows(), total=total, desc='Evaluating'):
    query_image, query_text, gt_video_path = get_covr_query(row)
    if query_image is None or not os.path.exists(gt_video_path):
        clip_ranks.append(None); agent_ranks.append(None)
        continue

    gt_norm = os.path.normpath(gt_video_path)
    gt_base = os.path.basename(gt_norm)

    # Stage 1 only (CLIP baseline)
    clip_res   = qa_system.coarse_search(
        query_image, query_text, corpus_embeddings, corpus_paths, top_n=TOP_K, alpha=ALPHA)
    clip_paths = [os.path.normpath(r[0]) for r in clip_res]
    if gt_norm in clip_paths:
        c_rank = clip_paths.index(gt_norm) + 1
    else:
        c_bases = [os.path.basename(p) for p in clip_paths]
        c_rank = (c_bases.index(gt_base) + 1) if gt_base in c_bases else None
    clip_ranks.append(c_rank)

    # Full Adaptive Retrieval (ReAgent-V)
    agent_res   = qa_system.adaptive_covr_retrieval(
        query_image, query_text, corpus_embeddings, corpus_paths,
        top_k=TOP_K, top_n_coarse=TOP_N_COARSE,
        max_iterations=MAX_ITERATIONS, reward_threshold=REWARD_THRESHOLD)
    agent_paths = [os.path.normpath(r[0]) for r in agent_res]
    if gt_norm in agent_paths:
        a_rank = agent_paths.index(gt_norm) + 1
    else:
        a_bases = [os.path.basename(p) for p in agent_paths]
        a_rank = (a_bases.index(gt_base) + 1) if gt_base in a_bases else None
    agent_ranks.append(a_rank)

    per_query.append({'idx': int(i), 'edit': query_text, 'gt': os.path.basename(gt_video_path),
                      'clip_rank': c_rank, 'agent_rank': a_rank})

elapsed = time.time() - t0
print(f'Done in {elapsed:.1f}s ({elapsed/total:.1f}s/query)')



In [ ]:
# In kết quả và lưu file
print('\n' + '=' * 70)
print('BENCHMARK RESULTS — ReAgent-V Composed Video Retrieval')
print('=' * 70)
print(f'{"Metric":<22} {"CLIP Only":>13} {"ReAgent-V Agent":>18} {"Delta":>10}')
print('-' * 70)

for k in [1, 5, 10]:
    rc = recall_at_k(clip_ranks, k, total) * 100
    ra = recall_at_k(agent_ranks, k, total) * 100
    d  = ra - rc
    print(f'  Recall@{k:<15} {rc:>12.2f}% {ra:>16.2f}%  {"+" if d>=0 else ""}{d:.2f}%')

mc = mrr(clip_ranks, total); ma = mrr(agent_ranks, total); dm = ma - mc
print(f'  {"MRR":<20} {mc:>13.4f} {ma:>18.4f}  {"+" if dm>=0 else ""}{dm:.4f}')
print('-' * 70)
print(f'  Evaluated on {total} queries | top_k={TOP_K} | max_iter={MAX_ITERATIONS}')
print('=' * 70)

summary = {
    'num_samples': total, 'top_k': TOP_K, 'max_iterations': MAX_ITERATIONS,
    'metrics': {
        'clip_only':      {'recall_at_1': recall_at_k(clip_ranks,1,total),  'recall_at_5': recall_at_k(clip_ranks,5,total),  'recall_at_10': recall_at_k(clip_ranks,10,total),  'mrr': mc},
        'reagentv_agent': {'recall_at_1': recall_at_k(agent_ranks,1,total), 'recall_at_5': recall_at_k(agent_ranks,5,total), 'recall_at_10': recall_at_k(agent_ranks,10,total), 'mrr': ma},
    },
    'per_query': per_query,
}
with open(RESULTS_PATH, 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)
print(f'Results saved → {RESULTS_PATH}')